# Somo la 07 - Mfumo wa Mpangilio wa Muundo

Daftari hili linaonyesha **Mfumo wa Mpangilio wa Muundo** kwa mawakala wa AI kwa kutumia Microsoft Agent Framework.
Utajifunza jinsi ya kuvunja maombi magumu ya safari kuwa kazi ndogo zilizo na muundo, kuzipeleka kwa mawakala maalum,
na kutekeleza mpangilio utakaotokeza — yote kwa kutumia matokeo yenye muundo yanayotolewa na mifano ya Pydantic.


## Usanidi


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Umegawaji wa Kazi

Umegawaji wa kazi ni msingi wa muundo wa upangaji. Badala ya kumuomba wakala mmoja kushughulikia ombi ngumu kutoka mwanzo hadi mwisho, tunagawanya tatizo kuwa **kazi ndogo** ndogo zilizounganishwa vyema. Kazi ndogo kila moja inatolewa kwa wakala maalum (kwa mfano, safari za ndege, hoteli, shughuli) kwa vipaumbele na mpangilio wa utegemezi ulio wazi.

Njia hii hutoa faida kadhaa:
- **Uwiano**: kila kazi ndogo ina jukumu moja.
- **Msimulizi**: kazi ndogo zisizotegemeana zinaweza kufanywa kwa wakati mmoja.
- **Uaminifu**: kushindwa kunahusiana tu na kazi ndogo binafsi.
- **Ufuatiliaji wa bajeti**: gharama zina kalkulika kwa kila kazi ndogo na kujumlishwa.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Kuunda Wakala wa Mipango na Matokeo Yaliyo Elekezwa

Wakala wa mipango hufanya kama **mratibu wa dawati la mbele**. Ikitolewa ombi la juu la kusafiri hutengeneza `TravelPlan` iliyopangwa — kugawanya ombi hilo kuwa kazi ndogo ndogo, kuweka vipaumbele, na kubainisha utegemezi ili concierge au safu ya utekelezaji iweze kufanya kazi hiyo.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Kutekeleza Mpango kwa Zana za Wataalamu

Mara afisa wa dawati la mbele anapotoa mpango uliopangwa, **afisa wa concierge** hutekeleza. Kila chombo cha mtaalamu hushughulikia aina moja ya kazi ndogo (ndege, hoteli, shughuli). Concierge hupitia kazi ndogo za mpango kwa mpangilio wa utegemezi na kutuma kila moja kwa chombo kinachofaa.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Muhtasari

Katika somo hili ulijifunza **Mfumo wa Mpango wa Ubunifu** kwa watoaji AI:

1. **Umegawaji wa Kazi** — Wakala wa mipango wa dawati la mbele hugawanya ombi tata la kusafiri kuwa kazi ndogo zilizo na muundo kwa kutumia mifano ya Pydantic, na kuzipa kila moja wakala maalum kwa vipaumbele na utegemezi.
2. **Matokeo Yenye Muundo** — Kwa kupitisha `response_format` wakala hurudisha kitu cha `TravelPlan` kilichothibitishwa badala ya maandishi ya kienyeji, kufanya usindikaji wa baadaye kuwa wa kuaminika.
3. **Utekelezaji wa Mpango** — Wakala msaidizi huzunguka kazi ndogo kwa kutumia zana maalum (`book_flight`, `reserve_hotel`, `book_activity`) kutekeleza mpango na kuripoti matokeo.

Mfano huu unatofautisha *kitu cha kufanya* (mpango) kutoka *jinsi ya kukifanya* (utekelezaji), na kuifanya wakala kuwa za kidijitali zaidi, rahisi kupimika, na rahisi kuongeza.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Sauti ya kujihadhari**:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya kutafsiri kwa AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kwa usahihi, tafadhali fahamu kuwa tafsiri za moja kwa moja zinaweza kuwa na makosa au upotovu wa taarifa. Nyaraka asilia kwa lugha yake ya asili inapaswa kuzingatiwa kama chanzo cha kuaminika. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatuna dhamana kwa kutoelewana au tafsiri isiyo sahihi inayotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
